# Minimal Pair practice with Garden Path Sentence
### COSC 426: Fall 2025, Colgate University

## Dataset & Minimal Pair Experiments

We need to answer the following question:

1. Is the garden path effect in gpt greater for long sentences, compared to short sentences?
2. Does utahnip/boola_gpt2_seed-1 generate more incorrect "yes" responses when followed by ambiguous sentences compared to unambiguous sentences?
3. Is the probability of incorrect "yes" proportional to the strength of the garden path effect?

To set up MinimalPair tsv:

1. We add a condition called `length` for the pair of sentences and use unambiguous sentence as expected. ambiguous sentence as unexpected.
2. We use the whole sentence with question and append the answer to it as the `sent`; we add `ROI` to only focus on the answer that the model predicts. Also we use `ambiguity` column to distinguish ambiguous and unambiguous sentences. For each pair "No." would be the expected and "Yes." is unexpected.
3. We use the same file from 2 and add a `length` column.

To make config files for NLPscholar:
1. Run analyze and evaluate mode and save by_cond (length).

2. Run analyze and evaluate mode and save by_pair. Optionally, we also save by_cond (ambiguity).

3. Run analyze and evaluate mode on both models and save by_cond (length) with prob as pred_measure.

## Results


#### Results from experiment 1

In [34]:
import pandas as pd

df = pd.read_csv('results/garden_1.tsv', sep='\t')
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
print(df)

df['abs_diff'] = df['diff'].abs()
long_penalty = df.loc[df['length'] == 'long', 'abs_diff'].iloc[0]
short_penalty = df.loc[df['length'] == 'short', 'abs_diff'].iloc[0]

print("\nGarden-path penalty (absolute surprisal difference):")
print(f"  Long sentences : {long_penalty:.4f}")
print(f"  Short sentences: {short_penalty:.4f}")

if short_penalty > long_penalty:
    print("\nConclusion: gpt2 shows a larger garden-path effect for SHORT sentences.")
else:
    print("\nConclusion: gpt2 shows a larger garden-path effect for LONG sentences.")

  model length  acc      diff  expected  unexpected  macrodiff
0  gpt2   long  1.0 -0.546824  7.057685    7.604509  -0.546824
1  gpt2  short  1.0 -0.672538  6.906247    7.578785  -0.672538

Garden-path penalty (absolute surprisal difference):
  Long sentences : 0.5468
  Short sentences: 0.6725

Conclusion: gpt2 shows a larger garden-path effect for SHORT sentences.


#### Results from experiment 2

In [14]:
df = pd.read_csv('results/garden_2.tsv', sep='\t')
df_bypair = pd.read_csv('results/garden_2_bypair.tsv', sep='\t')

num_incorrect_ambiguous = df_bypair.loc[
    (df_bypair['ambiguity'] == 'ambiguous') & (df_bypair['acc'] == 'False')
].shape[0]

num_incorrect_unambiguous = df_bypair.loc[
    (df_bypair['ambiguity'] == 'unambiguous') & (df_bypair['acc'] == 'False')
].shape[0]

print(f"  Ambiguous sentences   : {num_incorrect_ambiguous} incorrect")
print(f"  Unambiguous sentences : {num_incorrect_unambiguous} incorrect")

if num_incorrect_ambiguous > num_incorrect_ambiguous:
    print("\nConclusion: utahnlp/boolq_gpt2_seed-1 generate more incorrect answer when followed by ambiguous sentences.")
elif num_incorrect_unambiguous > num_incorrect_ambiguous:
    print("\nConclusion: utahnlp/boolq_gpt2_seed-1 generate more incorrect answer when followed by unambiguous sentences.")
else:
    print("\nConclusion: utahnlp/boolq_gpt2_seed-1 is equally likely to generate incorrect answers for both ambiguous and unambiguous sentences.")

  Ambiguous sentences   : 0 incorrect
  Unambiguous sentences : 0 incorrect

Conclusion: utahnlp/boolq_gpt2_seed-1 is equally likely to generate incorrect answers for both ambiguous and unambiguous sentences.


#### Results from experiment 3

In [30]:
df = pd.read_csv('results/garden_3.tsv', sep='\t')

df["incorrect_yes"] = df["unexpected"]

print(df[["model", "length", "diff", "incorrect_yes"]])

print("\nConclusion:")
for model in df['model'].unique():
    if df.loc[
        (df['length'] == 'short') & (df['model'] == model), 'diff'
    ].abs().iloc[0] > df.loc[
        (df['length'] == 'long') & (df['model'] == model), 'diff'
    ].abs().iloc[0]:
        print(f"\nFor {model}, the longer sentences have a smaller probability difference between expected and unexpected answer. ")
    else:
        print(f"\nFor {model}, the longer sentences have a larger probability difference between expected and unexpected answer. ")

    if df.loc[
        (df['length'] == 'short') & (df['model'] == model), 'incorrect_yes'
    ].iloc[0] > df.loc[
        (df['length'] == 'long') & (df['model'] == model), 'incorrect_yes'
    ].iloc[0]:
        print(f"For {model}, the shorter sentences lead to more incorrect 'yes' answers. ")
    else:
        print(f"For {model}, the longer sentences lead to more incorrect 'yes' answers. ")


                       model length      diff  incorrect_yes
0                       gpt2   long  0.029376       0.184855
1                       gpt2  short  0.032170       0.195903
2  utahnlp/boolq_gpt2_seed-1   long  0.066857       0.122870
3  utahnlp/boolq_gpt2_seed-1  short  0.111600       0.114389

Conclusion:

For gpt2, the longer sentences have a smaller probability difference between expected and unexpected answer. 
For gpt2, the shorter sentences lead to more incorrect 'yes' answers. 

For utahnlp/boolq_gpt2_seed-1, the longer sentences have a smaller probability difference between expected and unexpected answer. 
For utahnlp/boolq_gpt2_seed-1, the longer sentences lead to more incorrect 'yes' answers. 


## Discussion

Garden-path sentences introduce a temporary ambiguity that leads human comprehenders toward a highly preferred but incorrect parse. When disambiguating material arrives, humans typically experience processing difficulty. If language models were sensitive to the same structural pressures, we would expect:

- Higher predictability for the unexpected (garden-pathed).
- A larger gap between expected and unexpected sentences.
- More incorrect “yes” answers when ambiguity occurs.

Across all three analyses conducted in this study, however, neither GPT-2 nor BoolQ-GPT2 shows strong behavioral signatures of human-like garden-path confusion.

A primary limitation of this study is that the evaluation was conducted on a minimal working dataset and this dataset does not capture the full range of garden-path constructions found in natural language.

Prior studies demonstrate that language models’ sensitivity to garden-path ambiguity increases when tested on larger, more diverse stimulus sets. [BERT Shows Garden Path Effects](https://aclanthology.org/2023.eacl-main.235/) (Irwin et al., EACL 2023 highlights that fine-grained patterns emerge only when evaluating dozens of constructions across varied lexical environments. Likewise, [Garden Path Traversal in GPT-2](https://aclanthology.org/2022.blackboxnlp-1.25/) (Jurayj et al., BlackboxNLP 2022) shows that GPT-2’s structural expectations become clearer under broad-scale testing.

Because of these findings, it is possible that GPT-2 or BoolQ-GPT2 would show stronger or qualitatively different garden-path responses if tested on a richer and more varied dataset.